# Session 4 — Sampling Techniques

**Goal:** treat the registry as a *population* and draw samples from it under four
different designs, so you can see — measured, not asserted — how a sampling design
changes the answer you get, and tell the two kinds of error apart: **bias** (wrong in
the same direction every time) and **variance** (right on average, noisy each time).

## What this stage does for the system

Sessions 1-3 characterised the inputs. This one asks a question no amount of later
modelling can rescue: **is the registry itself a fair picture of the patients the
system will be used on?** Every number after this point — correlations, p-values,
regression coefficients, held-out AUC — is computed on this data and inherits whatever
distortion is baked into how it was collected.

The distinction that matters is which errors *shrink with more data*. Variance does:
collect four times as many patients and the noise halves. Bias does not — a sampling
design that systematically favours certain patients gives the same wrong answer at
n=60 and at n=60,000, and it does it with narrower and narrower confidence intervals,
so the estimate looks *more* trustworthy the more wrong data you gather. That failure
mode is invisible to every diagnostic in the rest of this module, which is why it is
worth a full session here.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.


For this session only, the 297 patients are treated as the whole **population** — a
convenient fiction that lets us draw samples from it and compare each estimate against
the true value we would otherwise never know.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — Set up the population and the quantity to estimate

Mean cholesterol is the target quantity throughout: one number, known exactly for the
population, that every sampling design below will try to recover from 60 patients.
The `clinic` column is constructed, not part of the UCI data — it stands in for
catchment areas that a real multi-site registry would have, and Step 6 needs it.

In [ ]:
import numpy as np

population = df.sort_values("age", kind="stable").reset_index(drop=True)

# Constructed: 10 "clinic sites", each serving a narrow age band. Real clinics cluster
# demographically for exactly this reason (catchment areas), which is the point.
population["clinic"] = (np.arange(len(population)) // 30).clip(max=9)

TRUE_MEAN = population["chol"].mean()
SAMPLE_SIZE = 60

print(f"population: {len(population)} patients")
print(f"TRUE mean cholesterol = {TRUE_MEAN:.2f} mg/dL   <- what every design below is trying to recover")
print()
print(population.groupby("clinic").agg(n=("age", "size"), mean_age=("age", "mean"),
                                       mean_chol=("chol", "mean")).round(1))

**Observe:** `TRUE mean cholesterol = 247.35`, and the per-clinic table showing mean
age rising `38.7 → 69.2` across the ten sites, with mean cholesterol drifting
`221.2 → 260.6` alongside it.
**Infer:** the clinics differ from each other on the very quantity being estimated —
which is what makes cluster sampling costly in Step 6 and what makes stratification
worth considering in Step 5. Note the mechanism: the clinics differ in cholesterol
*because* they differ in age, and age and cholesterol are correlated (Session 5 measures
this at `r = 0.20`). This is a construction, and it is labelled as one, but the
structure it imposes is the structure real multi-site data has.

## Step 3 — Simple random sampling: the honest baseline

Every patient has an equal chance of selection, drawn independently. SRS is unbiased
by construction — not because it is accurate on any single draw, but because it has no
systematic preference for any kind of patient.

In [ ]:
srs = population.sample(n=SAMPLE_SIZE, random_state=1)
estimate = srs["chol"].mean()

print(f"SRS estimate from {SAMPLE_SIZE} patients: {estimate:.2f} mg/dL")
print(f"true value:                          {TRUE_MEAN:.2f} mg/dL")
print(f"error on this draw:                  {estimate - TRUE_MEAN:+.2f} mg/dL")
print()
print(f"sample mean age: {srs['age'].mean():.1f}   (population: {population['age'].mean():.1f})")
print(f"sample disease rate: {srs['target'].mean():.3f}   (population: {population['target'].mean():.3f})")

**Observe:** an estimate about `3 mg/dL` below the truth, and — importantly — a sample
mean age of `56.0` against the population's `54.5`, with a disease rate of `0.533`
against `0.461`: both in the right neighbourhood *without anyone having asked them
to*.
**Infer:** that incidental match is the property that makes SRS the baseline: it is
unbiased for *every* quantity simultaneously, not just the one you happened to be
estimating. A design tuned to get cholesterol right can still distort age. The error on
this single draw is not the design's error though — one draw tells you nothing about a
design, which is why Step 7 repeats each design 500 times. Change `random_state` and
watch the estimate move by several mg/dL; that movement is variance, and Step 8
predicts its size from theory.

## Step 4 — Convenience sampling: the trap

Sampling whoever is easiest to reach. Here: patients under 50, standing in for
"whoever came to the daytime clinic" or "whoever the study coordinator could enrol
before Friday".

In [ ]:
under_50 = population[population["age"] < 50]
convenience = under_50.sample(n=SAMPLE_SIZE, random_state=1)
convenience_estimate = convenience["chol"].mean()

print(f"eligible pool: {len(under_50)} of {len(population)} patients")
print(f"convenience estimate: {convenience_estimate:.2f} mg/dL")
print(f"true value:           {TRUE_MEAN:.2f} mg/dL")
print(f"error:                {convenience_estimate - TRUE_MEAN:+.2f} mg/dL")
print()
print(f"sample mean age: {convenience['age'].mean():.1f}   (population: {population['age'].mean():.1f})")
print(f"sample disease rate: {convenience['target'].mean():.3f}   (population: {population['target'].mean():.3f})")

**Observe:** an error of `−13.27 mg/dL`, four times the SRS error — and a sample whose
mean age (`43.6` against `54.5`) and disease rate (`0.267` against `0.461`) are both
far below the population's.
**Infer:** the cholesterol error is a *symptom*; the disease-rate distortion is the
disease. A model trained on this sample would learn a baseline prevalence that does not
apply to the clinic it gets deployed in, and Session 1's Bayes calculation showed
exactly how much a wrong prevalence corrupts a predicted probability. The reason this
design is dangerous rather than merely inaccurate is in Step 7: its error does not
shrink with sample size, and — worse — its estimates are *less* scattered than SRS's,
so the usual "narrow interval means reliable estimate" instinct points the wrong way.

## Step 5 — Stratified sampling: using structure you already know

Split the population into strata, sample within each in proportion to its size. This
guarantees the sample's composition on the stratifying variable instead of leaving it
to chance — which reduces variance *if* the strata differ on the quantity being
estimated.

In [ ]:
def stratified_sample(frame, strata_col, n_total, seed):
    fraction = n_total / len(frame)
    return frame.groupby(strata_col, group_keys=False)[frame.columns].apply(
        lambda g: g.sample(n=round(len(g) * fraction), random_state=seed)
    )

stratified = stratified_sample(population, "target", SAMPLE_SIZE, seed=1)
stratified_estimate = stratified["chol"].mean()

print(f"stratified on `target`, n={len(stratified)}")
print(f"estimate: {stratified_estimate:.2f}   (true {TRUE_MEAN:.2f}, error {stratified_estimate - TRUE_MEAN:+.2f})")
print()
print("disease rate is now guaranteed, not left to chance:")
print(f"  sample: {stratified['target'].mean():.3f}    population: {population['target'].mean():.3f}")
print()
print("but do the strata actually differ on cholesterol?")
print(population.groupby("target")["chol"].agg(["mean", "std"]).round(2))

**Observe:** the sample's disease rate matches the population's almost exactly, but the
two strata's mean cholesterol (`243.49` vs `251.85`) are only 8 mg/dL apart — against a
within-stratum standard deviation above 49.
**Infer:** stratification here buys precision on `target` and essentially nothing on
`chol`, and Step 7 confirms it: stratified variance comes out no better than SRS. The
rule this illustrates is that stratification only reduces variance for quantities that
*differ between strata* — the gain comes from removing between-stratum variation, and
there is barely any here. Had we stratified on `age_group` instead, which relates to
cholesterol far more strongly, the gain would be real. This is the whole "how to
choose" of stratified sampling: pick the stratifying variable for its relationship to
what you are measuring, not for how natural a grouping it seems.

## Step 6 — Cluster sampling: cheaper per patient, at a precision cost

Sample whole pre-existing groups rather than individuals. Operationally far cheaper —
two clinic sites means two sets of approvals instead of sixty individual recruitments —
but patients within a site resemble each other, so 60 clustered patients carry less
information than 60 independent ones.

In [ ]:
def cluster_sample(frame, cluster_col, n_clusters, seed):
    chosen = frame[cluster_col].drop_duplicates().sample(n=n_clusters, random_state=seed)
    return frame[frame[cluster_col].isin(chosen)]

clustered = cluster_sample(population, "clinic", n_clusters=2, seed=1)
cluster_estimate = clustered["chol"].mean()

print(f"clinics selected: {sorted(clustered['clinic'].unique())}, n={len(clustered)} patients")
print(f"estimate: {cluster_estimate:.2f}   (true {TRUE_MEAN:.2f}, error {cluster_estimate - TRUE_MEAN:+.2f})")
print()
print("within-cluster homogeneity is the problem:")
print(f"  age sd within the sampled clinics: {clustered.groupby('clinic')['age'].std().mean():.2f}")
print(f"  age sd in the population:          {population['age'].std():.2f}")

**Observe:** the within-clinic age standard deviation, `2.14`, is under a quarter of
the population's `9.05`.
**Infer:** that ratio *is* the cost. Sixty patients drawn from two age-homogeneous
sites span a much narrower slice of the population than sixty drawn independently, so
the effective sample size is smaller than the nominal 60 — the "design effect".
Cluster sampling stays unbiased (the clinics are chosen at random), so this shows up
as extra variance in Step 7, not as a systematic shift. Which means it is honestly
reported by a correctly computed confidence interval — provided you compute one that
accounts for clustering. Feed clustered data to the ordinary formula in Step 9 and it
will hand back an interval that is too narrow.

## Step 7 — Repeat every design 500 times

One draw cannot distinguish a biased design from an unlucky one. Repeating each design
and looking at the *distribution* of its estimates separates the two: bias is where the
distribution is centred, variance is how wide it is.

In [ ]:
import matplotlib.pyplot as plt

n_trials = 500
designs = {"SRS": [], "Convenience (<50)": [], "Stratified (target)": [], "Cluster (2 clinics)": []}

for i in range(n_trials):
    designs["SRS"].append(population.sample(n=SAMPLE_SIZE, random_state=i)["chol"].mean())
    designs["Convenience (<50)"].append(under_50.sample(n=SAMPLE_SIZE, random_state=i)["chol"].mean())
    designs["Stratified (target)"].append(stratified_sample(population, "target", SAMPLE_SIZE, seed=i)["chol"].mean())
    designs["Cluster (2 clinics)"].append(cluster_sample(population, "clinic", 2, seed=i)["chol"].mean())

rows = []
for name, estimates in designs.items():
    e = np.array(estimates)
    rows.append({"design": name, "mean estimate": e.mean(), "bias": e.mean() - TRUE_MEAN,
                 "sd (variance)": e.std(), "RMSE": np.sqrt(((e - TRUE_MEAN) ** 2).mean())})
print(pd.DataFrame(rows).round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, estimates in designs.items():
    ax.hist(estimates, bins=30, alpha=0.5, label=name)
ax.axvline(TRUE_MEAN, color="black", ls="--", lw=2, label=f"truth = {TRUE_MEAN:.1f}")
ax.set_xlabel("estimated mean cholesterol (mg/dL)")
ax.set_ylabel("frequency across 500 samples")
ax.set_title("Where each design lands, over 500 repeats")
ax.legend()
plt.tight_layout()
plt.show()

**Observe:** SRS bias `+0.44` with sd `6.10`; convenience bias `−13.91` with sd
`3.05`; stratified bias `+0.45` with sd `6.40`; cluster bias `+0.25` with sd `8.84`.
In the histogram, the convenience distribution is the narrow one sitting well to the
left of the truth line.
**Infer:** the convenience row is the lesson of this session in two numbers: **the
lowest variance and the worst RMSE**. Its estimates agree with each other beautifully
and are all wrong the same way, so no amount of internal consistency reveals the
problem — and increasing n would shrink that `3.05` further while leaving the `−13.91`
untouched, making the estimate look ever more precise as it stays exactly as wrong.
Compare the cluster row: worse variance than SRS (`8.84` vs `6.10` — the design effect
from Step 6) but essentially no bias, which is a defensible trade for the operational
saving. And stratified confirms Step 5's prediction: no bias, no variance gain either,
because the strata barely differ on cholesterol.

## Step 8 — Standard error and the Central Limit Theorem

The variance seen in Step 7 is predictable, not mysterious. The **standard error** of
a sample mean is $\sigma/\sqrt{n}$ — and by the Central Limit Theorem, the sampling
distribution of that mean is approximately Normal *even when the underlying column is
not*, which is what rescues cholesterol from its skew problem in Session 3.

In [ ]:
sigma = population["chol"].std()
print(f"population sd = {sigma:.2f} mg/dL\n")
print(f"{'n':>5} {'theoretical SE':>16} {'to halve SE':>14}")
for n in [15, 60, 150, 297]:
    print(f"{n:>5} {sigma / np.sqrt(n):>16.2f} {4 * n:>14}")

print(f"\nSRS sd observed in Step 7: {np.std(designs['SRS']):.2f}")
print(f"Theory (sigma/sqrt(60)):   {sigma / np.sqrt(SAMPLE_SIZE):.2f}")

from scipy import stats as sps
sample_means = [population.sample(n=SAMPLE_SIZE, random_state=i)["chol"].mean() for i in range(500)]
print(f"\nskew of raw chol column:      {sps.skew(population['chol']):+.2f}")
print(f"skew of the 500 sample means: {sps.skew(sample_means):+.2f}   <- the CLT at work")

**Observe:** theoretical SE falls `13.43 → 6.71 → 4.25 → 3.02` as n grows; the observed
SRS sd from Step 7 is `6.10`, a little under the `6.71` theory predicts for n=60; and
the skew collapses from `+1.11` in the raw column to `+0.14` in the sample means.
**Infer:** two payoffs. First, the $\sqrt{n}$ is the reason data collection gets
expensive: halving the standard error costs *four times* the patients, as the third
column spells out — going from 60 to 240 to buy one more digit of precision. Second,
the skew collapse is the Central Limit Theorem made concrete on this exact column:
`chol` is emphatically not Normal (Session 3 rejected it at `p ≈ 1e-08`), yet the
distribution of its sample means is, which is precisely what licenses the t-based
interval in the next step and the t-tests in Sessions 6-7. And note what the SE formula
does *not* contain: any term for bias. It describes the spread of the convenience
design's estimates just as accurately as SRS's, while saying nothing about the fact
that they are centred 14 mg/dL from the truth.

The `6.10` versus `6.71` gap is not sampling noise either — it is the *finite
population correction*. Drawing 60 patients from a population of only 297 without
replacement is more informative than drawing 60 from an infinite one, by a factor of
$\sqrt{(N-n)/(N-1)} = 0.895$, and $6.71 \times 0.895 = 6.01$. The correction is
negligible whenever the sample is a small fraction of the population, which is the
usual case and the reason it is usually omitted; here the sample is a fifth of the
population, so it shows.

## Step 9 — Confidence intervals: a range instead of a point

A 95% confidence interval is the estimate plus or minus roughly two standard errors.
Its guarantee is about the *procedure*, not any one interval: repeated over many
samples, 95% of the intervals it produces contain the true value.

In [ ]:
from scipy import stats as sps

sample = population.sample(n=SAMPLE_SIZE, random_state=7)
mean = sample["chol"].mean()
se = sample["chol"].std(ddof=1) / np.sqrt(len(sample))
t_crit = sps.t.ppf(0.975, df=len(sample) - 1)

low, high = mean - t_crit * se, mean + t_crit * se
print(f"sample mean: {mean:.2f}   SE: {se:.2f}   t*: {t_crit:.3f}")
print(f"95% CI: [{low:.2f}, {high:.2f}]")
print(f"contains the true {TRUE_MEAN:.2f}? {low <= TRUE_MEAN <= high}")

# The guarantee is about the procedure -- check it by repeating.
covered = 0
for i in range(500):
    s = population.sample(n=SAMPLE_SIZE, random_state=i)["chol"]
    m, e = s.mean(), s.std(ddof=1) / np.sqrt(len(s))
    if m - t_crit * e <= TRUE_MEAN <= m + t_crit * e:
        covered += 1
print(f"\nSRS coverage over 500 intervals:  {covered / 500:.1%}   (target 95%)")

covered_conv = 0
for i in range(500):
    s = under_50.sample(n=SAMPLE_SIZE, random_state=i)["chol"]
    m, e = s.mean(), s.std(ddof=1) / np.sqrt(len(s))
    if m - t_crit * e <= TRUE_MEAN <= m + t_crit * e:
        covered_conv += 1
print(f"Convenience coverage:             {covered_conv / 500:.1%}   (target 95%)")

**Observe:** a single interval `[242.72, 269.12]` that does contain the truth, SRS
coverage of `97.4%` against the promised 95% — and convenience coverage of `21.8%`.
**Infer:** the convenience coverage number is the sharpest statement of this session.
Its intervals are *narrower* than SRS's (Step 7: sd `3.05` vs `6.10`) and yet they
capture the truth far less often, because a confidence interval only quantifies
variance and is silent about bias. Four in five of its intervals miss a value they
claim to capture 19 times in 20. (SRS erring the other way, at `97.4%`, is the finite
population correction again — the interval is slightly conservative because the formula
ignores that a fifth of the population was sampled.) Interval width therefore measures
how much data you collected, not how well you collected it — and the reflex of reading a tight interval
as a trustworthy estimate is exactly backwards for a biased design. Note also `ddof=1`
in the SE: with 60 patients the Session 2 denominator choice is no longer cosmetic.

## What this session hands to the next one

- **A validated registry.** Sessions 5-12 all proceed as if these 297 patients fairly
  represent the target population; this session is where that assumption gets stated
  rather than assumed, along with the caveat from Session 1 that this is a
  referral-clinic population.
- **The bias/variance distinction**, which is Session 12's entire subject — the same
  two error sources, relocated from sampling designs to model complexity.
- **Standard error and the CLT**, the machinery behind every p-value in Sessions 6-8.
- **The confidence interval**, and the discipline that it bounds noise only.

Session 5 takes the validated registry and asks the next question: which inputs
actually relate to disease, and to each other?

## Try it yourself

1. Change `SAMPLE_SIZE` to 150 and re-run Steps 7-9. Which numbers halve, which shrink
   by $\sqrt{2.5}$, and which do not move at all?
2. Stratify on `age_group` (Session 1's bins) instead of `target` in Step 5. Does the
   variance gain that `target` failed to deliver show up now?
3. In Step 6, sample 5 clinics instead of 2. How much of the design effect survives,
   and why does more clusters help more than more patients per cluster?
4. Build a second convenience design — say `sex == 1` only — and add it to Step 7. Is
   its bias on cholesterol larger or smaller than the age-based one, and would you have
   predicted the direction from Session 2's group means?